# 00 Data Audit and Cleaning


In [1]:
from pathlib import Path
import sys
sys.path.append(str(Path('..')/'src'))


## Objectives

- Load the raw dataset.
- Perform basic data quality checks (missingness, duplicates, weird categories).
- Clean + standardize key fields.
- Save a clean, modeling-ready dataset for all downstream notebooks.

**Output:** `data/processed/courses_clean.parquet`


In [2]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path('..') / 'data' / 'raw' / 'udemy_course_data.csv'
RAW_PATH.exists(), RAW_PATH


(True, WindowsPath('../data/raw/udemy_course_data.csv'))

In [3]:
df_raw = pd.read_csv(RAW_PATH)
df_raw.shape


(3683, 18)

In [4]:
df_raw.head(3)


,course_id,course_title,url,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject,profit,published_date,published_time,year,month,day
0,1070968,Ultimate Investment Banking Course,https://www.udemy.com/ultimate-investment-bank...,True,200,2147,23,51,All Levels,1.5 hours,2017-01-18T20:58:58Z,Business Finance,429400,2017-01-18,20:58:58Z,2017,1,18
1,1113822,Complete GST Course & Certification - Grow You...,https://www.udemy.com/goods-and-services-tax/,True,75,2792,923,274,All Levels,39 hours,2017-03-09T16:34:20Z,Business Finance,209400,2017-03-09,16:34:20Z,2017,3,9
2,1006314,Financial Modeling for Business Analysts and C...,https://www.udemy.com/financial-modeling-for-b...,True,45,2174,74,51,Intermediate Level,2.5 hours,2016-12-19T19:26:30Z,Business Finance,97830,2016-12-19,19:26:30Z,2016,12,19


In [5]:
# Basic schema
(df_raw.dtypes).to_frame('dtype')


,dtype
course_id,int64
course_title,object
url,object
is_paid,bool
price,int64
num_subscribers,int64
num_reviews,int64
num_lectures,int64
level,object
content_duration,object


In [6]:
# Missingness
missing = df_raw.isna().sum().sort_values(ascending=False)
missing[missing>0]


published_time    1
dtype: int64

In [7]:
# Duplicate course_id
n_unique = df_raw['course_id'].nunique()
num_dups = df_raw.duplicated('course_id').sum()
(n_unique, num_dups)


(3677, np.int64(6))

In [8]:
# Inspect noisy categories
print(df_raw['subject'].value_counts())
print(df_raw['level'].value_counts().head(10))
print(df_raw['content_duration'].head(5).tolist())


subject
Web Development        1200
Business Finance       1199
Musical Instruments     681
Graphic Design          603
Name: count, dtype: int64
level
All Levels            1932
Beginner Level        1271
Intermediate Level     421
Expert Level            58
52                       1
Name: count, dtype: int64
['1.5 hours', '39 hours', '2.5 hours', '3 hours', '2 hours']


## Cleaning

We use a small, deterministic cleaning function to:
- drop duplicated `course_id`
- normalize `level`
- parse `content_duration` to numeric hours
- parse timestamps


In [9]:
from course_rec.data import clean_courses

df = clean_courses(df_raw)
df.shape


(3677, 19)

In [10]:
df[['course_id','course_title','price','num_subscribers','num_reviews','num_lectures','content_duration','content_duration_hours','published_timestamp','subject','level']].head(5)


,course_id,course_title,price,num_subscribers,num_reviews,num_lectures,content_duration,content_duration_hours,published_timestamp,subject,level
0,1070968,Ultimate Investment Banking Course,200,2147,23,51,1.5 hours,1.5,2017-01-18 20:58:58+00:00,Business Finance,All Levels
1,1113822,Complete GST Course & Certification - Grow You...,75,2792,923,274,39 hours,39.0,2017-03-09 16:34:20+00:00,Business Finance,All Levels
2,1006314,Financial Modeling for Business Analysts and C...,45,2174,74,51,2.5 hours,2.5,2016-12-19 19:26:30+00:00,Business Finance,Intermediate Level
3,1210588,Beginner to Pro - Financial Analysis in Excel ...,95,2451,11,36,3 hours,3.0,2017-05-30 20:07:24+00:00,Business Finance,All Levels
4,1011058,How To Maximize Your Profits Trading Options,200,1276,45,26,2 hours,2.0,2016-12-13 14:57:18+00:00,Business Finance,Intermediate Level


In [11]:
# Post-clean checks
assert df['course_id'].is_unique
assert df['price'].min() >= 0
assert df['profit'].min() >= 0

(df.isna().sum().sort_values(ascending=False).head(10))


content_duration_hours    330
published_timestamp         1
course_id                   0
url                         0
course_title                0
num_subscribers             0
is_paid                     0
num_reviews                 0
num_lectures                0
level                       0
dtype: int64

In [12]:
# Save clean dataset
OUT_PATH = Path('..') / 'data' / 'processed' / 'courses_clean.parquet'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_PATH, index=False)
OUT_PATH


WindowsPath('../data/processed/courses_clean.parquet')